In [1]:
import torch
import torch.nn as nn
import requests
from PIL import Image

import warnings

# Ignore specific UserWarnings related to max_length in transformers
warnings.filterwarnings(
    "ignore", message=".*Using the model-agnostic default `max_length`.*"
)

In [2]:
class DummyModel(nn.Module):
    """
    A dummy model that consists of an embedding layer
    with two blocks of a linear layer followed by a layer
    norm layer.
    """

    def __init__(self):
        super().__init__()

        torch.manual_seed(123)

        self.token_embedding = nn.Embedding(2, 2)

        # Block 1
        self.linear_1 = nn.Linear(2, 2)
        self.layernorm_1 = nn.LayerNorm(2)

        # Block 2
        self.linear_2 = nn.Linear(2, 2)
        self.layernorm_2 = nn.LayerNorm(2)

        self.head = nn.Linear(2, 2)

    def forward(self, x):
        hidden_states = self.token_embedding(x)

        # Block 1
        hidden_states = self.linear_1(hidden_states)
        hidden_states = self.layernorm_1(hidden_states)

        # Block 2
        hidden_states = self.linear_2(hidden_states)
        hidden_states = self.layernorm_2(hidden_states)

        logits = self.head(hidden_states)
        return logits

In [3]:
def get_generation(model, processor, image, dtype):
    inputs = processor(image, return_tensors="pt").to(dtype)
    out = model.generate(**inputs)
    return processor.decode(out[0], skip_special_tokens=True)


def load_image(img_url):
    image = Image.open(requests.get(img_url, stream=True).raw).convert("RGB")

    return image

In [4]:
model = DummyModel()

In [6]:
def print_param_dtype(model):
    for name, param in model.named_parameters():
        print(f"{name} is loaded in {param.dtype}")


print_param_dtype(model)

token_embedding.weight is loaded in torch.float32
linear_1.weight is loaded in torch.float32
linear_1.bias is loaded in torch.float32
layernorm_1.weight is loaded in torch.float32
layernorm_1.bias is loaded in torch.float32
linear_2.weight is loaded in torch.float32
linear_2.bias is loaded in torch.float32
layernorm_2.weight is loaded in torch.float32
layernorm_2.bias is loaded in torch.float32
head.weight is loaded in torch.float32
head.bias is loaded in torch.float32


#### Cast the model into FP 16

In [7]:
from copy import deepcopy

model_fp16 = deepcopy(model)
model_fp16 = model_fp16.half()  # or model_fp16.to(torch.float16)

print_param_dtype(model_fp16)

token_embedding.weight is loaded in torch.float16
linear_1.weight is loaded in torch.float16
linear_1.bias is loaded in torch.float16
layernorm_1.weight is loaded in torch.float16
layernorm_1.bias is loaded in torch.float16
linear_2.weight is loaded in torch.float16
linear_2.bias is loaded in torch.float16
layernorm_2.weight is loaded in torch.float16
layernorm_2.bias is loaded in torch.float16
head.weight is loaded in torch.float16
head.bias is loaded in torch.float16


#### Cast the model into BFloat-16

In [10]:
from copy import deepcopy

model_bfp16 = deepcopy(model)
model_bfp16 = model_bfp16.to(torch.bfloat16)

print_param_dtype(model_bfp16)

token_embedding.weight is loaded in torch.bfloat16
linear_1.weight is loaded in torch.bfloat16
linear_1.bias is loaded in torch.bfloat16
layernorm_1.weight is loaded in torch.bfloat16
layernorm_1.bias is loaded in torch.bfloat16
linear_2.weight is loaded in torch.bfloat16
linear_2.bias is loaded in torch.bfloat16
layernorm_2.weight is loaded in torch.bfloat16
layernorm_2.bias is loaded in torch.bfloat16
head.weight is loaded in torch.bfloat16
head.bias is loaded in torch.bfloat16


In [12]:
dummy_input = torch.LongTensor([[1, 0], [0, 1]])

logits_fp32 = model(dummy_input)
logits_fp16 = model_fp16(dummy_input)
logits_bf16 = model_bfp16(dummy_input)

mean_diff = torch.abs(logits_fp16 - logits_fp32).mean().item()
max_diff = torch.abs(logits_fp16 - logits_fp32).max().item()

print(f"FP32 vs FP16 -- Mean diff: {mean_diff} | Max diff: {max_diff}")

mean_diff = torch.abs(logits_bf16 - logits_fp32).mean().item()
max_diff = torch.abs(logits_bf16 - logits_fp32).max().item()

print(f"FP32 vs BFP16 -- Mean diff: {mean_diff} | Max diff: {max_diff}")

FP32 vs FP16 -- Mean diff: 0.00020453333854675293 | Max diff: 0.00022590160369873047
FP32 vs BFP16 -- Mean diff: 0.0009979009628295898 | Max diff: 0.0016907453536987305


#### Get memory footprint

In [22]:
from transformers import AutoModel

model_path = "C:/Kartheek/Projects/PXXXX_NLP_Usecases_ProCodex_[none_US]/03_Project_Phase/Data_and_code/procodex_data/hf_models/bert-base-uncased"
model = AutoModel.from_pretrained(model_path)
model_bf16 = AutoModel.from_pretrained(model_path, torch_dtype=torch.bfloat16)

In [23]:
fp32_mem_footprint = model.get_memory_footprint()
print("Footprint of the fp32 model in bytes: ", fp32_mem_footprint)
print("Footprint of the fp32 model in MBs: ", fp32_mem_footprint / 1e6)

Footprint of the fp32 model in bytes:  437937152
Footprint of the fp32 model in MBs:  437.937152


In [25]:
bf16_mem_footprint = model_bf16.get_memory_footprint()
print("Footprint of the fp32 model in bytes: ", bf16_mem_footprint)
print("Footprint of the fp32 model in MBs: ", bf16_mem_footprint / 1e6)

Footprint of the fp32 model in bytes:  218972672
Footprint of the fp32 model in MBs:  218.972672


In [26]:
# Get the relative difference
relative_diff = bf16_mem_footprint / fp32_mem_footprint

print("Footprint of the bf16 model in MBs: ", bf16_mem_footprint / 1e6)
print(f"Relative diff: {relative_diff}")

Footprint of the bf16 model in MBs:  218.972672
Relative diff: 0.5000093529402136


#### Default Data Type

- For Hugging Face Transformers library, the deafult data type to load the models in is `float32`
- You can set the "default data type" as what you want.

In [27]:
desired_dtype = torch.bfloat16
torch.set_default_dtype(desired_dtype)

dummy_model_bf16 = DummyModel()
print_param_dtype(dummy_model_bf16)

# Reset
torch.set_default_dtype(torch.float32)

token_embedding.weight is loaded in torch.bfloat16
linear_1.weight is loaded in torch.bfloat16
linear_1.bias is loaded in torch.bfloat16
layernorm_1.weight is loaded in torch.bfloat16
layernorm_1.bias is loaded in torch.bfloat16
linear_2.weight is loaded in torch.bfloat16
linear_2.bias is loaded in torch.bfloat16
layernorm_2.weight is loaded in torch.bfloat16
layernorm_2.bias is loaded in torch.bfloat16
head.weight is loaded in torch.bfloat16
head.bias is loaded in torch.bfloat16
